# Josephson Current-Phase Relations

The **Josephson effect** describes dissipationless current flow between two
superconductors separated by a weak link. In S/F/S junctions — where a
ferromagnetic barrier replaces the conventional insulator — the interplay
between superconductivity and magnetism produces anomalous current-phase
relations, including the striking **0–$\pi$ transition**.

## Josephson Effect Basics

For a conventional S/I/S junction, the DC Josephson relation is:

$$
I(\phi) = I_c \sin\phi
$$

where $\phi = \varphi_L - \varphi_R$ is the phase difference between the two
superconducting electrodes and $I_c > 0$ is the critical current.

The ground state sits at $\phi = 0$ (zero phase difference, hence a
"**0-junction**"). The Josephson energy is:

$$
E_J(\phi) = -\frac{\Phi_0 I_c}{2\pi} \cos\phi
$$

where $\Phi_0 = h/(2e)$ is the flux quantum.

## S/F/S Junctions and the $\pi$-State

When the barrier is a ferromagnet, Cooper pairs acquire a finite center-of-mass
momentum $Q = 2E_{\text{ex}} / (\hbar v_F)$ due to the exchange splitting.
The pair amplitude oscillates across the F layer, picking up a
phase $\sim d_F / \xi_F$.

If this accumulated phase exceeds $\pi/2$, the effective critical current
changes sign, and the current-phase relation becomes:

$$
I(\phi) = |I_c| \sin(\phi + \pi) = -|I_c| \sin\phi
$$

The ground state is now at $\phi = \pi$ — a "**$\pi$-junction**".

### General Anomalous CPR

More generally, spin-orbit coupling or non-collinear magnetization can produce
an anomalous phase shift $\phi_0$:

$$
I(\phi) = I_c \sin(\phi + \phi_0)
$$

Such "$\phi_0$-junctions" spontaneously generate supercurrent at zero phase bias.

## Critical Current Oscillation

In the diffusive limit (Buzdin's result), the critical current of an S/F/S
junction oscillates and decays with the F-layer thickness:

$$
I_c(d_F) \propto \exp\!\left(-\frac{d_F}{\xi_F}\right)
\cos\!\left(\frac{d_F}{\xi_F} - \frac{\pi}{4}\right)
$$

where $\xi_F = \sqrt{\hbar D_F / E_{\text{ex}}}$ is the ferromagnetic
coherence length.

The oscillation nodes — where $I_c$ changes sign — mark the
**0–$\pi$ transitions**. The first transition occurs at:

$$
d_F^{0\text{-}\pi} \approx \frac{3\pi}{4}\, \xi_F \approx 0.75\, \xi_F
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import supermag
supermag.apply_theme("dark")

xi_F = 1.0  # normalize lengths to xi_F
d_F = np.linspace(0.01, 6.0, 500)

# Critical current (Buzdin formula)
Ic = np.exp(-d_F / xi_F) * np.cos(d_F / xi_F - np.pi / 4)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Left: Ic vs d_F
ax1.plot(d_F, Ic, 'b-', linewidth=2)
ax1.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax1.fill_between(d_F, 0, Ic, where=(Ic > 0), alpha=0.15, color='blue', label='0-state')
ax1.fill_between(d_F, 0, Ic, where=(Ic < 0), alpha=0.15, color='red', label='π-state')
ax1.set_xlabel(r'$d_F / \xi_F$', fontsize=13)
ax1.set_ylabel(r'$I_c$ (arb. units)', fontsize=13)
ax1.set_title('Critical current oscillation')
ax1.legend(fontsize=11)

# Right: CPR for 0 and π junctions
phi = np.linspace(-np.pi, 2*np.pi, 300)
ax2.plot(phi/np.pi, np.sin(phi), 'b-', linewidth=2, label='0-junction')
ax2.plot(phi/np.pi, -np.sin(phi), 'r--', linewidth=2, label='π-junction')
ax2.set_xlabel(r'$\phi / \pi$', fontsize=13)
ax2.set_ylabel(r'$I / I_c$', fontsize=13)
ax2.set_title('Current-phase relation')
ax2.legend(fontsize=11)
ax2.axhline(0, color='gray', linestyle='--', linewidth=0.8)

fig.tight_layout()
plt.show()

### SUPERMag Josephson Solver

The `supermag.josephson.current_phase_relation()` function computes $I(\phi)$
for an S/F/S junction via the C++ composable path.  Below we compute CPRs
for several F-layer thicknesses, demonstrating the 0–π transition detected
automatically from the sign of $I_c$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import supermag

xi_F = 1.0   # nm
E_ex = 256.0 # meV (Fe)
T    = 4.0   # K

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: CPR for several d_F values spanning the 0→π transition
d_F_values = [0.5, 1.5, 2.36, 3.5, 5.0]
for d_F in d_F_values:
    phi, I = supermag.josephson.current_phase_relation(d_F, xi_F, E_ex, T, n_phases=200)
    junction_type = '0' if I[len(I)//4] > 0 else 'π'
    axes[0].plot(phi / np.pi, I, lw=2, label=rf'$d_F$={d_F} nm ({junction_type})')

axes[0].axhline(0, ls='--', color='grey', alpha=0.5)
axes[0].set_xlabel(r'$\phi / \pi$', fontsize=13)
axes[0].set_ylabel(r'$I / I_c$', fontsize=13)
axes[0].set_title('CPR via supermag.josephson (C++ path)', fontsize=14)
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# Right: critical current Ic(d_F) from the API
d_F_sweep = np.linspace(0.3, 8.0, 80)
Ic_arr = np.zeros(len(d_F_sweep))
for i, dF in enumerate(d_F_sweep):
    phi_i, I_i = supermag.josephson.current_phase_relation(dF, xi_F, E_ex, T, n_phases=100)
    # Critical current = max of I(φ) near φ = π/2
    Ic_arr[i] = I_i[len(I_i)//4]   # I(π/2) carries the sign

axes[1].plot(d_F_sweep, Ic_arr, 'b-', lw=2)
axes[1].axhline(0, ls='--', color='grey', alpha=0.5)
axes[1].fill_between(d_F_sweep, 0, Ic_arr, where=(Ic_arr > 0),
                      alpha=0.15, color='blue', label='0-state')
axes[1].fill_between(d_F_sweep, 0, Ic_arr, where=(Ic_arr < 0),
                      alpha=0.15, color='red', label='π-state')
axes[1].set_xlabel(r'$d_F$ (nm)', fontsize=13)
axes[1].set_ylabel(r'$I_c$ (normalised)', fontsize=13)
axes[1].set_title(r'$I_c(d_F)$ — 0–π transition (API)', fontsize=14)
axes[1].legend(fontsize=11); axes[1].grid(True, alpha=0.3)

fig.tight_layout(); plt.show()

In [ ]:
# Josephson energy landscape: 0, π, and φ₀ junctions
import numpy as np
import matplotlib.pyplot as plt

phi = np.linspace(-np.pi, 3*np.pi, 500)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

configs = [
    (r'0-junction', 0.0, 'blue'),
    (r'$\pi$-junction', np.pi, 'red'),
    (r'$\phi_0$-junction ($\phi_0=\pi/3$)', np.pi/3, 'green'),
]

for ax, (label, phi0, color) in zip(axes, configs):
    E_J = -np.cos(phi - phi0)     # E_J / (Φ₀ Ic / 2π)
    I = np.sin(phi - phi0)
    ax.plot(phi/np.pi, E_J, color=color, lw=2, label=r'$E_J(\phi)$')
    ax.plot(phi/np.pi, I, color=color, lw=1.5, ls='--', alpha=0.6, label=r'$I(\phi)$')
    # Mark ground state
    phi_gs = phi0
    ax.axvline(phi_gs/np.pi, ls=':', color='grey', alpha=0.5)
    ax.set_xlabel(r'$\phi / \pi$', fontsize=12)
    ax.set_title(label, fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel(r'$E_J$, $I$ (normalised)', fontsize=12)
fig.tight_layout(); plt.show()

In [ ]:
# Temperature-driven 0-π transition (Ryazanov-type)
import numpy as np
import matplotlib.pyplot as plt

# Model: ξ_F(T) depends on temperature through E_ex(T) ~ E_ex(0)(1 - (T/T_Curie)^2)
# At fixed d_F, changing T changes the effective ξ_F → crosses 0-π boundary

T = np.linspace(0.3, 9.0, 400)   # K
Tc = 9.2; E_ex_0 = 256            # meV
d_F = 2.0                         # nm (fixed)

D_F_nm2_K = 3.04e5                # hbar*D/k_B in nm^2·K
E_ex_K = E_ex_0 * 11.6            # meV → K

# Simplified: at each T, compute Ic(T) ∝ exp(-d_F/ξ_F) cos(d_F/ξ_F - π/4) × Δ(T)/Δ(0)
xi_F_T = np.sqrt(D_F_nm2_K / E_ex_K)   # ~ constant (exchange >> T)
# Add weak T-dependence through BCS gap
Delta_T = 1.764 * Tc * np.sqrt(np.maximum(1 - T/Tc, 0))

Ic_T = Delta_T * np.exp(-d_F / xi_F_T) * np.cos(d_F / xi_F_T - np.pi/4)

# For the Ryazanov model, use a range of d_F values to show d_F-driven transition
d_F_arr = np.linspace(0.1, 6, 400)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: Ic vs d_F at two temperatures
for T_val, col, ls in [(2.0, 'blue', '-'), (7.0, 'red', '--')]:
    Delta = 1.764 * Tc * np.sqrt(max(1 - T_val/Tc, 0))
    Ic = Delta * np.exp(-d_F_arr / xi_F_T) * np.cos(d_F_arr / xi_F_T - np.pi/4)
    axes[0].plot(d_F_arr, Ic / np.max(np.abs(Ic)), col, ls=ls, lw=2,
                 label=f'$T$ = {T_val} K')
axes[0].axhline(0, ls='--', color='grey', alpha=0.5)
axes[0].set_xlabel(r'$d_F$ (nm)', fontsize=13)
axes[0].set_ylabel(r'$I_c$ (normalised)', fontsize=13)
axes[0].set_title(r'$I_c(d_F)$ at two temperatures', fontsize=14)
axes[0].legend(fontsize=11); axes[0].grid(True, alpha=0.3)

# Right: SQUID diffraction pattern with π-junction shift
Phi_ext = np.linspace(-3, 3, 500)  # in units of Φ₀
Ic_squid_0 = np.abs(np.cos(np.pi * Phi_ext))         # 0-0 SQUID
Ic_squid_pi = np.abs(np.cos(np.pi * Phi_ext + np.pi/2))  # 0-π SQUID (shifted by Φ₀/2)
axes[1].plot(Phi_ext, Ic_squid_0, 'b-', lw=2, label='0-0 SQUID')
axes[1].plot(Phi_ext, Ic_squid_pi, 'r--', lw=2, label=r'0-$\pi$ SQUID')
axes[1].set_xlabel(r'$\Phi_{\rm ext} / \Phi_0$', fontsize=13)
axes[1].set_ylabel(r'$I_c^{\rm SQUID}$ (normalised)', fontsize=13)
axes[1].set_title(r'SQUID diffraction: $\Phi_0/2$ shift', fontsize=14)
axes[1].legend(fontsize=11); axes[1].grid(True, alpha=0.3)

fig.tight_layout(); plt.show()

## Experimental Signatures

The $\pi$-junction state has been observed experimentally in several systems:

- **Nb/CuNi/Nb** — Ryazanov et al. (2001): temperature-driven 0–$\pi$ transition
- **Nb/PdNi/Nb** — Kontos et al. (2002): thickness-driven transition
- **SQUID half-quantum flux** — inserting a $\pi$-junction in a superconducting
  loop shifts the diffraction pattern by $\Phi_0/2$

The $\pi$-junction has potential applications as a phase battery in
superconducting qubits and as a bias-free flux source.

## References

1. Buzdin, A.I., "Proximity effects in superconductor-ferromagnet heterostructures," Rev. Mod. Phys. **77**, 935 (2005).
2. Golubov, A.A., Kupriyanov, M.Yu. & Il'ichev, E., "The current-phase relation in Josephson junctions," Rev. Mod. Phys. **76**, 411 (2004).
3. Ryazanov, V.V. et al., "Coupling of two superconductors through a ferromagnet," Phys. Rev. Lett. **86**, 2427 (2001).